In [ ]:
import sys
sys.path.append('/home/lumargot/trachoma/src/py')

import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # put -1 to not use any

In [ ]:
import math
import pandas as pd
import numpy as np 
import seaborn as sns
from utils import remove_labels

In [ ]:
import SimpleITK as sitk
import torch
import matplotlib.pyplot as plt

In [ ]:
# df_train = pd.read_csv('/CMF/data/lumargot/trachoma/mtss_seg.csv')
df_train = pd.read_csv('/CMF/data/lumargot/trachoma/csv_mtss_pret/csv_updated/mtss_pret_combined.csv')
df_train = df_train.loc[df_train['dataset'] !='Pret']

In [ ]:
seg_column='seg'
class_column='class'
mount_point='/CMF/data/lumargot/trachoma'
drop_labels=['Reject', 'Short Incision']
concat_labels=['overcorrection', 'Gap', 'ECA', 'Fleshy']
label_column = 'label'

In [ ]:
df_train = remove_labels(df_train, class_column, label_column, drop_labels=drop_labels, concat_labels=concat_labels)

In [ ]:
df_train['class'] = df_train['class'] -1 
df_train[['class', 'label']].value_counts()

In [ ]:
df_unique_subjects= df_train.drop_duplicates(subset='filename')

In [ ]:
all_filenames = df_unique_subjects['filename'].to_list()

In [ ]:
l_left, l_middle, l_right = [], [], []
for filename in all_filenames:

  df_subject = df_train.loc[ df_train['filename'] == filename]

  seg_path = filename.replace('img', 'seg_old').replace('.jpg', '.nrrd')
  seg_path = os.path.join(mount_point, seg_path)
  img_path = os.path.join(mount_point, filename)


  img= sitk.GetArrayFromImage(sitk.ReadImage(img_path)).copy()
  seg_t = torch.tensor(sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).copy())
  seg_t = (seg_t == 3).float()

  bb = compute_eye_bbx(seg_t).numpy()

  seg_cropped = seg_t[bb[1]:bb[3],bb[0]:bb[2]].numpy()
  img_cropped = img[bb[1]:bb[3],bb[0]:bb[2]]

  # plt.figure(figsize=(12,5))
  # plt.subplot(121)
  # plt.imshow(img_cropped)

  # plt.subplot(122)
  # plt.title('cropped seg')
  # plt.imshow(seg_cropped)

  # plt.tight_layout()
  # plt.show()

  x = df_subject['x_patch'] - bb[0]
  y = df_subject['y_patch'] - bb[1]
  labels = df_subject['class']
  h,w = seg_cropped.shape
  
  portion_side = w/4

  outcome_left, outcome_middle, outcome_right = get_outcome_per_section(x, labels, portion_side)
  l_left.append(outcome_left)
  l_middle.append(outcome_middle)
  l_right.append(outcome_right)

In [ ]:
df_unique_subjects['outcome_left'] = l_left
df_unique_subjects['outcome_middle'] = l_middle
df_unique_subjects['outcome_right'] = l_right

In [ ]:
df_unique_subjects

In [ ]:
df_unique_subjects[['cid', 'filename', 'dataset', 'outcome_left', 'outcome_middle', 'outcome_right']].to_csv('/CMF/data/lumargot/trachoma/PoPP_Data/mtss/outcomes_3_sections_v2.csv')